In [5]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. FUNCIÓN PARA INICIAR NAVEGADOR
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"  # comentar si estás en Windows
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

# ==========================================================
# 2. CONFIGURACIÓN
# ==========================================================
URL_OBJETIVO = "https://books.toscrape.com/"
SELECTOR_CONTENEDOR = "article.product_pod"
SELECTOR_TITULO = "h3 a"
SELECTOR_PRECIO = "p.price_color"

paginas_objetivo = 5

# ==========================================================
# 3. EJECUCIÓN
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print("Iniciando scraping...")
    driver.get(URL_OBJETIVO)

    for i in range(paginas_objetivo):
        print(f"📄 Procesando página {i + 1}")

        # Esperar que carguen los productos
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
        )

        productos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)

        for item in productos:
            try:
                # EXTRAER DATOS
                titulo = item.find_element(By.CSS_SELECTOR, SELECTOR_TITULO).get_attribute("title")
                precio = item.find_element(By.CSS_SELECTOR, SELECTOR_PRECIO).text

                # LIMPIAR PRECIO
                precio_limpio = "".join(filter(str.isdigit, precio))

                dataset_final.append({
                    "titulo": titulo,
                    "precio": precio_limpio,
                    "precio_num": valor_limpio,
                    "pagina": i + 1
                })

            except Exception:
                continue

        # INTENTAR IR A LA SIGUIENTE PÁGINA
        try:
            boton_siguiente = driver.find_element(By.XPATH, "//li[@class='next']/a")
            boton_siguiente.click()
            time.sleep(2)

        except Exception:
            print("Última página alcanzada.")
            break

    # ==========================================================
    # 4. GUARDAR DATOS
    # ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_scraping_completo.csv"
        df.to_csv(nombre_archivo, index=False)

        print("Proceso exitoso. Total registros: {len(dataset_final)}")
        print("Archivo generado: {nombre_archivo}")
        print(df.head())

    else:
        print("No se capturaron datos.")

except Exception as e:
    print(f"Error durante ejecución: {e}")

finally:
    driver.quit()
    print("Navegador cerrado.")

Iniciando scraping...
📄 Procesando página 1
📄 Procesando página 2
📄 Procesando página 3
📄 Procesando página 4
📄 Procesando página 5
Proceso exitoso. Total registros: {len(dataset_final)}
Archivo generado: {nombre_archivo}
                                  titulo precio  pagina
0                   A Light in the Attic   5177       1
1                     Tipping the Velvet   5374       1
2                             Soumission   5010       1
3                          Sharp Objects   4782       1
4  Sapiens: A Brief History of Humankind   5423       1
Navegador cerrado.
